In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import svd, orth, qr
from scipy.optimize import curve_fit
from numpy import sin, cos, pi
cmap = plt.cm.plasma
from scipy.ndimage import gaussian_filter1d
from scipy.signal import wiener
from scipy.signal import wiener, butter, freqs, convolve, freqz
from scipy.fft import fft, ifft, fftshift, ifftshift, fftfreq
from scipy.interpolate import (RegularGridInterpolator, RectBivariateSpline,
                               interpn, griddata, Rbf, interp1d, interp2d)
import cv2
from scipy.signal import convolve2d as conv2
from skimage import color, data, restoration
from smooth_POD_ROM.reduced_order_model import pulse, delta_n_width, train_ROM, L2_error, zielfunktion
from smooth_POD_ROM.pre_processing import convolve_f, gaussian_f, gaussian, smoothen
from smooth_POD_ROM.post_processing import richardson_lucy, post_process
from scipy.optimize import minimize

In [2]:
import matplotlib.pyplot as plt
import matplotlib as mpl
page_width_pt = 455.24
pt2in = 0.01389 
pt2cm = 0.0352777778
cm2in = 1/2.54
page_width_cm = 13.70499
# TODO: work with textwidth
plot_width_in = page_width_pt*pt2in/2
page_width_in = page_width_cm*cm2in
print(plot_width_in/cm2in)

fs = 10
fs_lbl = 6
plt.rcParams["figure.figsize"] = (plot_width_in, plot_width_in/1.61803398875)
plt.rcParams["figure.autolayout"] = True
plt.rcParams['font.size'] = fs
plt.rcParams['axes.titlesize'] = fs
plt.rcParams['axes.labelsize'] = fs
plt.rcParams['xtick.labelsize'] = fs
plt.rcParams['ytick.labelsize'] = fs
plt.rcParams['legend.labelspacing'] = 0.0
plt.rcParams['legend.fontsize'] = fs_lbl
plt.rcParams['legend.handlelength'] = 2.0

plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
mpl.rc('text', usetex=False)
mpl.rc('font', family='serif', size=fs, serif='Computer Modern Roman')
pth = "../Plots/"

markevery = 40

8.030570172000001


In [3]:
m, n = 1500, 10

x = np.linspace(0, 1, m, endpoint=False)
mu_val_ = x.copy()[::4]

shape = (m,)
clip = True
n_iter = 200

NN = np.arange(3, 150, 2)
n_test = 9
sigma_opt = 1/(NN*3)
c_opt = 1/(NN/50)*0 + 1.3


In [4]:
eROM = np.zeros_like(NN, dtype=np.float64)
esROM = np.zeros_like(NN, dtype=np.float64)
esROMs100 = np.zeros_like(NN, dtype=np.float64)
esROMs250 = np.zeros_like(NN, dtype=np.float64)
esROMs50 = np.zeros_like(NN, dtype=np.float64)


print("sigma", "c", "mean_ROM", "mean_sROM", "mean_sROMs", "improvement")

for i, n in enumerate(NN):
    print(n)
    rank = n

    # generate training data:
    X_train = np.empty((m, n))
    mu_train = np.linspace(0, 1, n, endpoint=True)[:, None]#
    for j, mu_j in enumerate(mu_train):
        X_train[:, j] = pulse(x, mu_j)
    # validation data:
#     interpolatable = (mu_train[0]<mu_val_) & (mu_val_<mu_train[-1])
#     mu_val = mu_val_[interpolatable][:, None]
#     X_val = np.empty((m, len(mu_val)))
#     for j, mu_j in enumerate(mu_val):
#         X_val[:, j] = pulse(x, mu_j)
    # generate test data
    n1 = n//2
    
    mu_test = np.linspace(mu_train[n1, 0], mu_train[n1+1, 0], n_test, endpoint=False)[:, None]
    X_test = np.empty((m, n_test))
    for j, mu_j in enumerate(mu_test):
        X_test[:, j] = pulse(x, mu_j)


    # optimize using test set
    x0 = np.array([sigma_opt[i], c_opt[i]])
    zf =  lambda params: zielfunktion(params, x, mu_train, X_train, mu_test, X_test, rank, shape, n_iter)[0]
    #res = minimize(zf, x0, method='nelder-mead', bounds=[(0, 1), (-1, 10)], options={'disp': True, "fatol": 1e-2})
    #res = minimize(zf, x0, method='Powell', bounds=[(0, 1), (-1, 10)], options={'disp': True, "fatol": 1e-2})
    #res = minimize(zf, x0, method='BFGS', options={'disp': True, "eps": np.array([.001, 0.1]), "maxiter": 50})
    #res = minimize(zf, x0, method='COBYLA', options={'disp': True, "rhobeg": np.array([.025, 0.5]), "maxiter": 50})
    res = minimize(zf, x0, method='SLSQP', bounds=[(0.001, 10*sigma_opt[i]), (-1, 10*c_opt[i])],
                   options={'disp': True, "eps": np.array([.0005, 0.05]), "maxiter": 25, "ftol": 0.0005})
#     res = minimize(zf, x0, method='trust-exact', bounds=[(0.001, 10*sigma_opt[i]), (-1, 10*c_opt[i])],
#                options={'disp': True, "eps": np.array([.0005, 0.05]), "maxiter": 25, "ftol": 0.0005, "max_trust_radius": np.array([.0005, 0.05])})
#     bounds = Bounds([0.001, -1.], [10*sigma_opt[i], 10*c_opt[i]])
#     result = direct(zf, bounds)
    print(res["message"])
    
    sigma_opt[i], c_opt[i] = res["x"]
    # TODO: run with mu_val?
    _i, _R, _sR, _sRs = zielfunktion(res["x"], x, mu_train, X_train, mu_test, X_test, rank, shape, 100)
    eROM[i] = np.mean(L2_error(_R, X_test))
    esROM[i] = np.mean(L2_error(_sR, X_test))
    esROMs100[i] = np.mean(L2_error(_sRs, X_test))
    _i, _R, _sR, _sRs = zielfunktion(res["x"], x, mu_train, X_train, mu_test, X_test, rank, shape, 250)
    esROMs250[i] = np.mean(L2_error(_sRs, X_test))
    _i, _R, _sR, _sRs = zielfunktion(res["x"], x, mu_train, X_train, mu_test, X_test, rank, shape, 50)
    esROMs50[i] = np.mean(L2_error(_sRs, X_test))
    print(eROM[i], esROM[i], esROMs100[i], esROMs250[i])


sigma c mean_ROM mean_sROM mean_sROMs improvement
3
0.11111111, 1.30000000, 

/home/florianma@ad.ife.no/matpro_files/Florian/Repositoties/POD-ROM-and-gaussian-convolution/src/smooth_POD_ROM/post_processing.py:24: UserWarning: image needs to be 2D
  warnings.warn("image needs to be 2D")


0.26814362, 0.25800616, 0.28429893, 6.0249 %
0.11161111, 1.30000000, 0.26814362, 0.25800095, 0.28424896, 6.0062 %
0.11111111, 1.35000000, 0.26814362, 0.25800616, 0.28430874, 6.0285 %
1.11111111, 1.22679577, 0.26814362, 0.26220219, 0.26220219, -2.2158 %
1.11061111, 1.22679577, 0.26814362, 0.26220219, 0.26220219, -2.2158 %
1.11111111, 1.27679577, 0.26814362, 0.26220219, 0.26220219, -2.2158 %
Optimization terminated successfully    (Exit mode 0)
            Current function value: -2.215765238969098
            Iterations: 2
            Function evaluations: 6
            Gradient evaluations: 2
Optimization terminated successfully
1.11111111, 1.22679577, 0.26814362, 0.26220219, 0.26220219, -2.2158 %
1.11111111, 1.22679577, 0.26814362, 0.26220219, 0.26220219, -2.2158 %
1.11111111, 1.22679577, 0.26814362, 0.26220219, 0.26220219, -2.2158 %
0.2681436235501976 0.2622021903510274 0.26220219034981296 0.2622021903487893
5
0.06666667, 1.30000000, 0.26917355, 0.23499706, 0.28676497, 6.5353 %
0.067

0.00301667, 12.99998680, 0.21184821, 0.21154236, 0.25294856, 19.4008 %
0.03005666, 8.98943653, 0.21184821, 0.19311072, 0.04671770, -77.9476 %
0.03275625, 8.58903609, 0.21184821, 0.19328910, 0.04493610, -78.7885 %
0.03325625, 8.58903609, 0.21184821, 0.19335830, 0.04506240, -78.7289 %
0.03275625, 8.63903609, 0.21184821, 0.19328910, 0.04496769, -78.7736 %
0.00329768, 12.96832792, 0.21184821, 0.21130140, 0.25540185, 20.5589 %
0.02981039, 9.02696527, 0.21184821, 0.19311192, 0.04700938, -77.8099 %
0.03246166, 8.63282901, 0.21184821, 0.19325338, 0.04499223, -78.7620 %
0.03268932, 8.59898515, 0.21184821, 0.19328066, 0.04494007, -78.7867 %
0.03273190, 8.59265545, 0.21184821, 0.19328601, 0.04493699, -78.7881 %
0.03274636, 8.59050584, 0.21184821, 0.19328784, 0.04493638, -78.7884 %
0.03275205, 8.58966038, 0.21184821, 0.19328857, 0.04493620, -78.7885 %
0.03275444, 8.58930538, 0.21184821, 0.19328887, 0.04493614, -78.7885 %
0.03275546, 8.58915302, 0.21184821, 0.19328900, 0.04493612, -78.7885 %
0.0327

/home/florianma@ad.ife.no/yes/lib/python3.8/site-packages/scipy/optimize/_optimize.py:353: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "


...

0.00100000, -1.00000000, 0.05063406, 0.04768455, 0.04720099, -6.7802 %
0.00175349, 2.64463911, 0.05063406, 0.04675518, 0.05344464, 5.5508 %
0.00209129, 4.27856279, 0.05063406, 0.04685842, 0.02906372, -42.6005 %
0.00259129, 4.27856279, 0.05063406, 0.04746769, 0.03296273, -34.9001 %
0.00209129, 4.32856279, 0.05063406, 0.04685842, 0.02901577, -42.6952 %
0.00207826, 4.36525590, 0.05063406, 0.04684936, 0.02906176, -42.6043 %
0.00208471, 4.32237410, 0.05063406, 0.04685380, 0.02906105, -42.6057 %
0.00208790, 4.30111339, 0.05063406, 0.04685603, 0.02906211, -42.6036 %
0.00208954, 4.29022829, 0.05063406, 0.04685718, 0.02906279, -42.6023 %
0.00209038, 4.28462313, 0.05063406, 0.04685778, 0.02906320, -42.6015 %
0.00209082, 4.28171881, 0.05063406, 0.04685809, 0.02906344, -42.6010 %
0.00209104, 4.28020922, 0.05063406, 0.04685825, 0.02906357, -42.6007 %
0.00209116, 4.27942242, 0.05063406, 0.04685833, 0.02906364, -42.6006 %
0.00209122, 4.27901177, 0.05063406, 0.04685838, 0.02906368, -42.6005 %
0.002091

0.00853166, 12.99998850, 0.04903565, 0.06555907, 0.08770923, 78.8683 %
0.00529782, 6.50079657, 0.04903565, 0.05446381, 0.07772190, 58.5008 %
0.00377125, 3.43279350, 0.04903565, 0.04931270, 0.04140446, -15.5625 %
0.00324673, 2.37862702, 0.04903565, 0.04775023, 0.02757533, -43.7647 %
0.00374673, 2.37862702, 0.04903565, 0.04923582, 0.03072415, -37.3432 %
0.00324673, 2.42862702, 0.04903565, 0.04775023, 0.02785798, -43.1883 %
0.00588761, -0.99999984, 0.04903565, 0.05652865, 0.04692140, -4.3117 %
0.00351081, 2.04076433, 0.04903565, 0.04851375, 0.02662258, -45.7077 %
0.00401081, 2.04076433, 0.04903565, 0.05007857, 0.02930571, -40.2359 %
0.00351081, 2.09076433, 0.04903565, 0.04851375, 0.02704073, -44.8550 %
0.00104720, -0.99999994, 0.04903565, 0.04586667, 0.04531957, -7.5783 %
0.00268061, 1.01607279, 0.04903565, 0.04634094, 0.04726727, -3.6063 %
0.00335024, 1.84256882, 0.04903565, 0.04804309, 0.02492060, -49.1786 %
0.00385024, 1.84256882, 0.04903565, 0.04956231, 0.02681689, -45.3114 %
0.003350

0.00308398, 1.30000000, 0.04831018, 0.04672232, 0.03011936, -37.6542 %
0.00258398, 1.35000000, 0.04831018, 0.04547393, 0.04196509, -13.1341 %
0.02583979, 13.00000000, 0.04831018, 0.11254855, 0.04766613, -1.3332 %
0.01414133, 7.11450069, 0.04831018, 0.08237141, 0.08229658, 70.3504 %
0.00783638, 3.94248256, 0.04831018, 0.06304169, 0.07483911, 54.9138 %
0.00481181, 2.42082082, 0.04831018, 0.05239927, 0.03742358, -22.5348 %
0.00378323, 1.90334149, 0.04831018, 0.04886326, 0.02484502, -48.5719 %
0.00428323, 1.90334149, 0.04831018, 0.05054876, 0.02691827, -44.2803 %
0.00378323, 1.95334149, 0.04831018, 0.04886326, 0.02522388, -47.7877 %
0.00598071, -1.00000000, 0.04831018, 0.05656627, 0.04667598, -3.3827 %
0.00419107, 1.36449059, 0.04831018, 0.05023202, 0.02131907, -55.8704 %
0.00469107, 1.36449059, 0.04831018, 0.05197269, 0.02414123, -50.0287 %
0.00419107, 1.41449059, 0.04831018, 0.05023202, 0.02184195, -54.7881 %
0.00646799, -1.00000000, 0.04831018, 0.05829451, 0.04773411, -1.1924 %
0.004544

0.00341776, 1.88098835, 0.04594725, 0.04661145, 0.02177622, -52.6060 %
0.00341851, 1.88007196, 0.04594725, 0.04661400, 0.02177473, -52.6093 %
0.00341862, 1.87993533, 0.04594725, 0.04661438, 0.02177450, -52.6098 %
0.00341864, 1.87991497, 0.04594725, 0.04661444, 0.02177447, -52.6099 %
0.00341864, 1.87991193, 0.04594725, 0.04661445, 0.02177446, -52.6099 %
0.00341864, 1.87991148, 0.04594725, 0.04661445, 0.02177446, -52.6099 %
0.00341864, 1.87991141, 0.04594725, 0.04661445, 0.02177446, -52.6099 %
Optimization terminated successfully    (Exit mode 0)
            Current function value: -52.60986534048762
            Iterations: 2
            Function evaluations: 21
            Gradient evaluations: 2
Optimization terminated successfully
0.00341864, 1.87991141, 0.04594725, 0.04661445, 0.02285341, -50.2616 %
0.00341864, 1.87991141, 0.04594725, 0.04661445, 0.02163331, -52.9171 %
0.00341864, 1.87991141, 0.04594725, 0.04661445, 0.02508640, -45.4017 %
0.045947249092993386 0.04661444747119606 0.02

0.00345791, 1.46944494, 0.04517439, 0.04627130, 0.02323984, -48.5553 %
0.00395791, 1.46944494, 0.04517439, 0.04809539, 0.02553860, -43.4666 %
0.00345791, 1.51944494, 0.04517439, 0.04627130, 0.02351052, -47.9561 %
0.00100085, 2.70579486, 0.04517439, 0.04200265, 0.05353635, 18.5104 %
0.00321221, 1.59307993, 0.04517439, 0.04541481, 0.02326520, -48.4991 %
0.00334148, 1.52803435, 0.04517439, 0.04586117, 0.02323248, -48.5716 %
0.00339767, 1.49976138, 0.04517439, 0.04605820, 0.02323236, -48.5718 %
0.00342565, 1.48567892, 0.04517439, 0.04615696, 0.02323520, -48.5655 %
0.00344044, 1.47823576, 0.04517439, 0.04620932, 0.02323719, -48.5611 %
0.00344841, 1.47422759, 0.04517439, 0.04623756, 0.02323836, -48.5586 %
0.00345273, 1.47205297, 0.04517439, 0.04625289, 0.02323902, -48.5571 %
0.00345509, 1.47086802, 0.04517439, 0.04626125, 0.02323939, -48.5563 %
0.00345637, 1.47022129, 0.04517439, 0.04626582, 0.02323959, -48.5558 %
0.00345707, 1.46986841, 0.04517439, 0.04626831, 0.02323970, -48.5556 %
Optimiz

In [5]:
pth = "C:/Users/florianma/OneDrive - Institutt for Energiteknikk/Documents/convolution_paper/data/"
pth = "/home/florianma@ad.ife.no/matpro_files/Florian/results/sPODROM/"

In [6]:
# eROM[i] = np.mean(L2_error(_R, X_test))
# esROM[i] = np.mean(L2_error(_sR, X_test))
# esROMs[i] = np.mean(L2_error(_sRs, X_test))
# esROMs
np.save(pth+"eROM.npy", eROM)
np.save(pth+"esROM.npy", esROM)
np.save(pth+"esROMs20.npy", esROMs50)
np.save(pth+"esROMs100.npy", esROMs100)
np.save(pth+"esROMs250.npy", esROMs250)
np.save(pth+"sigma_opt200.npy", sigma_opt)
np.save(pth+"c_opt200.npy", c_opt)

In [7]:
print(NN.shape, s50.shape)

NameError: name 's50' is not defined

In [ ]:
s50 = np.load(pth+"sigma_opt50.npy")
s200 = np.load(pth+"sigma_opt200.npy")
s100 = np.load(pth+"sigma_opt.npy")

eROM200 = np.load(pth+"eROM200.npy")
esROM200 = np.load(pth+"esROM200.npy")
esROMs200 = np.load(pth+"esROMs200.npy")
sigma_opt200 = np.load(pth+"sigma_opt200.npy")
c_opt200 = np.load(pth+"c_opt200.npy")



plt.plot(NN, s50, "C0.")
plt.plot(NN, s100[::2], "C1.")
plt.plot(NN, s200, "C2.")
plt.ylim(0, 0.02)

In [ ]:
fig, ax = plt.subplots(figsize=(page_width_in/2, page_width_in/3))
plt.plot(NN, eROM, "C0.-", ms=2, label="$u_{rb}$")
#plt.plot(NN, esROM, "C1--", ms=2, label="$u_{rb,S}$")
plt.plot(NN, esROMs100, "C2.-", ms=2, label="$u_{\mu,rb,D_{100}}$")
plt.plot(NN, esROMs250, "C3.-", ms=2, label="$u_{\mu,rb,D_{250}}$")
#plt.plot(mu_val, esROMs2, "C3<-", ms=4, markevery=(20, markevery), label="$u_{rb,D_{100}}$")
plt.legend()
plt.ylabel("$\|u_{\mu,rb}-u_{\mu}\|_{L_2}$")
plt.xlabel("$N$")
# plt.ylim(0, 0.4)
plt.legend()
ax.set_xticks(np.linspace(0, 250, 26, endpoint=True), minor=True)
ax.set_yticks(np.linspace(0, .4, 19, endpoint=True), minor=True)
plt.grid(True, which='minor', linestyle='--', lw=.25)
plt.grid(True, which='major', linestyle='-')
ax.set_yscale('log')
plt.xlim(0, 150)
plt.ylim(2e-2, .3)
plt.show()

In [ ]:

fig, ax = plt.subplots(figsize=(page_width_in/2, page_width_in/3))
plt.plot(NN, eROM, "C0o-", ms=4, label="$u_{rb}$")
plt.plot(NN, esROM, "C1--", ms=4, label="$u_{rb,S}$")
plt.plot(NN, esROMs100, "C2.-", ms=4, label="$u_{rb,D_{100}}$")
plt.plot(NN, esROMs250, "C3.-", ms=4, label="$u_{rb,D_{250}}$")
#plt.plot(mu_val, esROMs2, "C3<-", ms=4, markevery=(20, markevery), label="$u_{rb,D_{100}}$")
plt.legend()
plt.ylabel("$\|u_{rb}-u_h\|_{L_2}$")
plt.xlabel("$N$")
# plt.ylim(0, 0.4)
plt.legend()
ax.set_xticks(np.linspace(0, 250, 26, endpoint=True), minor=True)
ax.set_yticks(np.linspace(0, .4, 19, endpoint=True), minor=True)
plt.grid(True, which='minor', linestyle='--', lw=.25)
plt.grid(True, which='major', linestyle='-')
ax.set_yscale('log')
plt.xlim(0, 70)
plt.ylim(1e-2, 1)
plt.show()

In [ ]:

fig, ax = plt.subplots(figsize=(page_width_in/2, page_width_in/3))
ax2 = ax.twinx()
ax.plot(NN, sigma_opt, "C0o-", ms=4, label="$\sigma_S$")
ax2.plot(NN, c_opt, "C1--", ms=4, label="$c$")
#plt.plot(mu_val, esROMs2, "C3<-", ms=4, markevery=(20, markevery), label="$u_{rb,D_{100}}$")
plt.legend()
plt.ylabel("$\sigma_S, c$")
plt.xlabel("$N$")
plt.legend()
ax.set_xticks(np.linspace(0, 250, 26, endpoint=True), minor=True)
ax.set_yticks(np.linspace(0, .4, 19, endpoint=True), minor=True)
plt.grid(True, which='minor', linestyle='--', lw=.25)
plt.grid(True, which='major', linestyle='-')
plt.xlim(0, 100)

plt.show()

In [ ]:

fig, ax = plt.subplots(figsize=(page_width_in/2, page_width_in/3))
ax2 = ax.twinx()
ax.plot(NN, sigma_opt, "C0o-", ms=4, label="$\sigma_S$")
ax2.plot(NN, c_opt, "C1--", ms=4, label="$c$")
#plt.plot(mu_val, esROMs2, "C3<-", ms=4, markevery=(20, markevery), label="$u_{rb,D_{100}}$")
plt.legend()
plt.ylabel("$\sigma_S, c$")
plt.xlabel("$N$")
plt.legend()
ax.set_xticks(np.linspace(0, 250, 26, endpoint=True), minor=True)
ax.set_yticks(np.linspace(0, .4, 19, endpoint=True), minor=True)
plt.grid(True, which='minor', linestyle='--', lw=.25)
plt.grid(True, which='major', linestyle='-')
plt.xlim(0, 100)

plt.show()

In [ ]:
overlap = 0.075*NN / 1
plt.plot(NN, c_opt, "C0o")
plt.plot(NN, 1/(NN**.5/10), "C1.")
#plt.ylim(0, 0.1)

In [ ]:
overlap = 0.075*NN / 1
plt.plot(NN, sigma_opt, "C0.")
plt.plot(NN, 1/(NN*4), "C1.")
plt.ylim(0, .1)

In [ ]:
overlap = 0.075*NN / 1
plt.plot(NN, c_opt, "C0.")
plt.plot(NN, 50/(NN), "C1.")
plt.ylim(0, 2)